# Week 1 analysis - classic DNS vs DoH latency

**Purpose:** first look at your own measurement data. If something is fundamentally wrong (empty columns, every query failing, DoH impossibly fast) you find out now, not in week 4.

Reads every `results/raw_*.csv` (both teammates' files, after `git pull`).

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# find the repo root (the folder that contains client/) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "client").is_dir())
for sub in ("client", "ml", "capture"):
    sys.path.insert(0, str(ROOT / sub))
RESULTS = ROOT / "results"
print("repo root:", ROOT)

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from summarize import load_raw, summarize, summarize_by_run
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

df = load_raw(RESULTS)
print(len(df), "rows from", df.source_file.nunique(), "files;  people:", sorted(df.person.unique()),
      " profiles:", sorted(df.profile.unique()))
df.head()

## 1. Did the queries work?
`status` should be mostly `ok`. Timeouts/errors are real data - report them, do not delete them.

In [ ]:
(df.groupby(["resolver", "method"]).status.value_counts().unstack(fill_value=0))

## 2. Latency distribution per method and resolver
Boxplots show median, spread and outliers. Outliers (retries, slow resolvers) are *kept* - report mean AND median.

In [ ]:
ok = df[df.status == "ok"]
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=ok, x="resolver", y="rtt_ms", hue="method", ax=ax, showfliers=True, fliersize=2)
ax.set_ylabel("RTT (ms)"); ax.set_xlabel("resolver"); ax.set_title("Resolution latency by method")
fig.tight_layout(); fig.savefig(RESULTS / "fig_week1_rtt_boxplot.png", dpi=300); plt.show()

## 3. Numbers for the paper
`summarize` = over all queries; `summarize_by_run` = mean of each run first, then mean ± std across runs (the honest one).

In [ ]:
summarize(df).round(2)

In [ ]:
summarize_by_run(df).round(2)

## 4. Do the two locations differ?
Different ISPs/paths between teammates is itself a useful robustness signal.

In [ ]:
summarize(df, by=["person", "resolver", "method"])[["person","resolver","method","n_ok","mean_ms","median_ms","p95_ms"]].round(2)

## 5. Privacy: what can a passive observer read?
The latency numbers above are only half of the trade-off. `capture/privacy_audit.py` searches your pcap for the domains you queried: classic DNS should expose ~100 %, DoH ~0 %.

```
python capture/privacy_audit.py capture/pcaps/<file>.pcap --client "results/raw_A_clean_r1_*.csv" --out results/privacy_audit_A.csv
```

In [ ]:
files = sorted(RESULTS.glob("privacy_audit_*.csv"))
if files:
    pa = pd.concat([pd.read_csv(f).assign(file=f.name) for f in files])
    display(pa[["file", "channel", "domains_queried", "domains_visible_to_observer", "exposure_rate", "sni_seen"]].round(3))
else:
    print("no privacy_audit_*.csv yet - run capture/privacy_audit.py (see above)")